# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walk-through for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all dataset entities by their `@id`.

### Dataset Source
The dataset's Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Get metadata as dict (using .to_json())
metadata = dataset.metadata.to_json()
print(f"Dataset: {metadata['name']}")
print(metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant model organizes tables as 'record sets', each with fields defined by their unique `@id`.

Let's list all record sets, their `@id`s, and then for each, enumerate their fields/columns by `@id`.

In [ ]:
# Extract all record sets and their fields' @ids
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in metadata.\nPlease check if the dataset exposes data as record sets in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}")
        print(f"  Name: {rs.get('name','<no name>')}")
        print(f"  Description: {rs.get('description','<no description>')}")
        print(f"  Fields:")
        for field in rs.get('field', []):
            # Each field has its own @id
            if isinstance(field, dict):
                print(f"    - {field.get('@id', '<unknown id>')} ({field.get('name', '')})")
            else:
                print(f"    - {field}")
        print()
# Save record set ids for later cells
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference all record sets and fields by their `@id`.

If multiple record sets exist, we'll demonstrate loading the first. Adjust `record_set_id` as needed.

In [ ]:
# Extract data from all available record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if record_set_ids:
    record_set_id_example = record_set_ids[0]
    print(f"Columns in record set '{record_set_id_example}':")
    print(dataframes[record_set_id_example].columns.tolist())
    display(dataframes[record_set_id_example].head())
else:
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data.

**Note:** For demonstration, we'll pick a numeric field by its `@id` if available. Replace variables below with those matching your schema.

In [ ]:
# EDA depends on the schema; adapt the field '@id's as appropriate for your data

import numpy as np

if record_set_ids:
    # Choose a dataset and pick first available numeric field
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    # Heuristically pick a numeric field by inspecting dtypes
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].quantile(0.80) # Use 80th percentile as a meaningful threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize the numeric field (Z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Pick a non-numeric/grouping field for demonstration if available
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
            print(f"Grouped filtered data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found in DataFrame.")
else:
    print("No record sets present.")

## 5. Visualization
Visualize distributions or relationships using the extracted and processed fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated loading and exploring the FAIR² dataset using `mlcroissant`, with all referencing by `@id`. We inspected the dataset's metadata, enumerated available record sets and fields, ingested records as DataFrames, and performed basic EDA and visualization.

Adapt the field `@id`s in the templates above to reference the most analytically relevant variables for your analysis. For in-depth usage, consult the dataset's Croissant schema for a full listing of entity `@id` values.